In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

import matplotlib 
matplotlib.rc('xtick', labelsize=20) 
matplotlib.rc('ytick', labelsize=20)

from sklearn.model_selection import train_test_split
from spectraltools import Spectral
from tensorflow import keras
from tensorflow.keras.models import save_model #saveweights
from tensorflow.keras.models import load_model

In [ ]:
delimiter = ' '

# Load the file
df_spectrum = pd.read_csv('miles_stars.csv')
df_spectrum.columns = df_spectrum.iloc[0]

# Drop the first row as it's now used as header
df_spectrum = df_spectrum[1:]

# Reset the index
df_spectrum.reset_index(drop=True, inplace=True)

wavelength = df_spectrum['wav'].astype(float)
star_id = 's0050'
star_spectrum = df_spectrum[star_id].astype(float)

file_path = 'params.ascii'
df_params = pd.read_csv(file_path, delimiter=' ')
df_params.set_index('ID', inplace=True)## Data Cleaning


## Data Cleaning

In [ ]:
# Remove the first and the last part of the spectrum where the instrument does not return values of intensity
df_spectrum = df_spectrum.iloc[50:len(df_spectrum.iloc[:,0])-30, :]
wavelength = df_spectrum['wav'].astype(float)

star_spectrum = df_spectrum[star_id].astype(float)
df_spectrum = df_spectrum.astype(float)
df_spectrum = df_spectrum.iloc[:,:-1]

## Normalization

In [ ]:
# Normalize the columns
normalized_df =df_spectrum.div(df_spectrum.max())
df_spectrum = normalized_df

In [ ]:
#remove nan and inf
nan_indices = df_params.index[df_params['__Fe_H_'].isna()]
inf_indices = df_params.index[np.isinf(df_params['__Fe_H_'])]
indices_to_drop = df_spectrum.T.index[nan_indices]
df_spectrum_metallicity = df_spectrum.T.drop(indices_to_drop).T
cleaned_data_metallicity = df_params['__Fe_H_'].replace([np.inf, -np.inf], np.nan).dropna()

max_metallicity = max(np.abs(cleaned_data_metallicity))
cleaned_data_metallicity = cleaned_data_metallicity / max_metallicity

df_spectrum_metallicity = df_spectrum_metallicity.T

X = df_spectrum_metallicity
y = cleaned_data_metallicity

# Spectral Model

In [ ]:
# SPECTRAL NN MODEL

def spectral_model():
    mod = tf.keras.Sequential()
    mod.add(tf.keras.Input(shape=(x_train.shape[1])))
    mod.add(Spectral(units=200,
                     is_base_trainable=True,
                     is_diag_end_trainable=False,
                     is_diag_start_trainable=True,
                     activation='relu',
                     use_bias=True,
                     diag_end_initializer='Zeros',
                     diag_start_initializer='Ones',
                     diag_regularizer=tf.keras.regularizers.L2(l2=1e-2),
                     base_regularizer=tf.keras.regularizers.L2(l2=1e-4)))
    mod.add(tf.keras.layers.BatchNormalization())
    mod.add(tf.keras.layers.Dropout(0.01))
    mod.add(Spectral(units=50,
                     is_base_trainable=True,
                     is_diag_end_trainable=False,
                     is_diag_start_trainable=True,
                     activation='tanh',
                     use_bias=True,
                     diag_end_initializer='Zeros',
                     diag_start_initializer='Ones'))
#     mod.add(tf.keras.layers.Dropout(0.1))
    mod.add(Spectral(units=50,
                     is_base_trainable=True,
                     is_diag_end_trainable=False,
                     is_diag_start_trainable=True,
                     activation='tanh',
                     use_bias=True,
                     diag_end_initializer='Zeros',
                     diag_start_initializer='Ones'))
#     mod.add(tf.keras.layers.Dropout(0.1))
    mod.add(Spectral(1,
                     is_base_trainable=True,
                     is_diag_end_trainable=False,
                     is_diag_start_trainable=True,
                     activation='tanh',
                     use_bias=True,
                     diag_end_initializer='Zeros',
                     diag_start_initializer='Ones'))
    return mod


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
mod = spectral_model()
mod.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='mae',
    metrics='mae',
)

hist = mod.fit(x_train, y_train,
               batch_size=10,
               epochs=1000,
               validation_split=0.1,
               verbose=1)

eig_start = Spectral.return_diag(mod.layers[0])
base_start = mod.layers[0].base
norma = tf.linalg.norm(base_start, ord=2, axis=1)
eig_norm = (eig_start * norma)

Eigenvalues distribution

In [ ]:
plt.figure(figsize=(10,8))
plt.hist(lambda_mean/np.max(lambda_mean), 50, facecolor='tab:orange', ec = 'grey')
plt.yscale('log')
plt.show()

Loss Vs number of spectral lines used as input of the model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_prune = []
mse_prune = []

eig_norm = eig_norm.numpy()

max_features = 500 

for n in range(1,max_features):
    mask = eig_norm.copy()
    th = np.sort(np.abs(eig_norm))[-n]
    mask[np.abs(mask)<th]=0
    mask[np.abs(mask)>=th]=1
    y_pred = mod.predict(x_test*mask, verbose=0)
    
    mae_prune.append(mean_absolute_error(y_test, y_pred))
    mse_prune.append(mean_squared_error(y_test, y_pred)) 

xM = 200
vl = 31
plt.figure(figsize=(7,6))
plt.plot(np.arange(1, max_features)[:xM]/len(eig_norm), mae_prune[:xM], color='tab:olive',marker='o')
plt.xlabel('fraction of components used', fontsize=23)
plt.ylabel('MAE', fontsize=23)
plt.show()


An example of stellar spectra comparaed with the network eigenvalues

In [ ]:
star_id = 's0050'
star_spectrum = df_spectrum[star_id].astype(float)

plt.figure(figsize=(10,6))

plt.axvline(3934, ls ='dotted', color = 'r')  # CaII H
plt.axvline(3968, ls ='dotted', color = 'r')  # K
plt.axvline(5173, ls ='dotted', color = 'r')  # Mg_1
plt.axvline(5184, ls ='dotted', color = 'r')  # Mg_2
plt.axvline(5270, ls ='dotted', color = 'r')  # Fe_1
plt.axvline(5896, ls ='dotted', color = 'r')  # Na_D_1
plt.axvline(5890, ls ='dotted', color = 'r')  # Na_D_2
plt.axvline(4667, ls ='dotted', color = 'r')  # FeIII(?)
plt.axvline(5018, ls ='dotted', color = 'r')  # FeII(?)
plt.axvline(4384, ls ='dotted', color = 'r')  # FeI
plt.axvline(4481, ls ='dotted', color = 'r')  # MaII(?)
plt.axvline(4861, ls ='dotted', color = 'r')  # Balmer_7

plt.plot(wavelength, star_spectrum, 'darkorange', label = 'Spectrum')
plt.plot(wavelength, np.abs(eig_norm), 'b', label = '$\lambda_i$')
plt.legend(fontsize = 15)
plt.xlabel('Wavelength', fontsize='18')
plt.ylabel('Intensity', fontsize='18')
plt.show()